# Week 3 — UTM Academic Advisor Data Pipeline
**Pathway A: PDF/TXT document data**

This notebook implements ingestion, inspection, PDF text extraction, chunking comparison, preparation, validation, storage, SHA-256 lineage and a repeatability smoke test. It does not call an LLM or create embeddings/vector indexes.

## Before running
1. Download approved UTM regulation PDFs from the URLs in `docs/week3_data_inventory.md`.
2. Confirm the current edition and terms of use.
3. Run the cells in order. Upload PDFs when prompted.
4. Review empty-text pages: scanned pages may require OCR and are not silently treated as ingested.

Prepared outputs are written to `/content/week3_project/data/processed/`. Colab runtime storage is temporary; download the outputs or copy them to an authorised persistent location after execution.


In [ ]:
# Cell 1 — Initialise workspace and dependencies (run first)
from pathlib import Path
import sys, subprocess, json, hashlib, re, shutil
from datetime import datetime, timezone

BASE_DIR = Path('/content/week3_project')
RAW_DIR = BASE_DIR / 'data' / 'raw'
PROCESSED_DIR = BASE_DIR / 'data' / 'processed'
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

try:
    from pypdf import PdfReader
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pypdf'])
    from pypdf import PdfReader

print('Raw folder:', RAW_DIR)
print('Processed folder:', PROCESSED_DIR)
print('pypdf ready')


In [ ]:
# Cell 2 — Upload source documents into data/raw
from google.colab import files
uploaded = files.upload()
for filename, payload in uploaded.items():
    target = RAW_DIR / Path(filename).name
    target.write_bytes(payload)
    print('Saved:', target.name, '| bytes:', target.stat().st_size)
print('If files were already copied into data/raw, you may skip uploading and continue.')


In [ ]:
# Cell 3 — Verify ingestion and select exact sources
all_files = sorted(p for p in RAW_DIR.rglob('*') if p.is_file())
print('Input file count:', len(all_files))
for p in all_files:
    print(p.relative_to(RAW_DIR), '| bytes:', p.stat().st_size)

pdf_files = [p for p in all_files if p.suffix.lower() == '.pdf']
txt_files = [p for p in all_files if p.suffix.lower() == '.txt']
source_files = pdf_files + txt_files
if not source_files:
    raise FileNotFoundError('No PDF/TXT sources found. Upload approved source documents to data/raw and rerun this cell.')
print('Selected PDF/TXT sources:')
for i, p in enumerate(source_files): print(i, p.name)
assert len(source_files) >= 1, 'At least one source is required; two are recommended for comparison.'


In [ ]:
# Cell 4 — Inspect content and compare chunking approaches
def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

def extract_pages(path):
    path = Path(path)
    if path.suffix.lower() == '.pdf':
        reader = PdfReader(str(path))
        return [(i + 1, page.extract_text() or '') for i, page in enumerate(reader.pages)], len(reader.pages)
    return [(1, path.read_text(encoding='utf-8', errors='replace'))], 1

def clean_text(text):
    text = text.replace('\\x00', ' ')
    text = re.sub(r'[ \\t]+', ' ', text)
    text = re.sub(r'\\n{3,}', '\\n\\n', text)
    return text.strip()

def fixed_chunks(text, size=900):
    return [text[i:i+size].strip() for i in range(0, len(text), size) if text[i:i+size].strip()]

def paragraph_chunks(text, size=900):
    paragraphs = [p.strip() for p in re.split(r'\\n\\s*\\n', text) if p.strip()]
    chunks, current = [], ''
    for para in paragraphs:
        if len(para) > size:
            if current: chunks.append(current); current = ''
            chunks.extend(fixed_chunks(para, size))
        elif not current:
            current = para
        elif len(current) + 2 + len(para) <= size:
            current += '\\n\\n' + para
        else:
            chunks.append(current); current = para
    if current: chunks.append(current)
    return chunks

documents = []
for src in source_files:
    pages, page_count = extract_pages(src)
    cleaned_pages = [(page_no, clean_text(text)) for page_no, text in pages]
    empty_pages = [page_no for page_no, text in cleaned_pages if not text]
    full_text = '\\n\\n'.join(text for _, text in cleaned_pages if text)
    if not full_text:
        print('WARNING: no extractable text:', src.name, '| empty pages:', empty_pages[:20])
    fixed_count = sum(len(fixed_chunks(text)) for _, text in cleaned_pages if text)
    para_count = sum(len(paragraph_chunks(text)) for _, text in cleaned_pages if text)
    documents.append({'path': src, 'pages': cleaned_pages, 'page_count': page_count, 'empty_pages': empty_pages, 'full_text': full_text, 'fixed_count': fixed_count, 'paragraph_count': para_count})
    print('\\nSOURCE:', src.name)
    print('SHA-256:', sha256_file(src))
    print('Pages:', page_count, '| extracted characters:', len(full_text), '| empty-text pages:', empty_pages)
    print('Fixed chunks:', fixed_count, '| paragraph-aware chunks:', para_count)
    print('Sample:', repr(full_text[:350]))
    if not full_text: print('ACTION: inspect this PDF; scanned pages may need OCR before use.')

assert any(d['full_text'] for d in documents), 'No source produced extractable text. Review PDFs/OCR before continuing.'


In [ ]:
# Cell 5 — Prepare page-aware paragraph chunks and preserve provenance
CHUNK_SIZE = 900
CHOSEN_METHOD = 'paragraph-aware-char-900-per-page'
records = []
for doc in documents:
    src = doc['path']
    source_id = 'S' + str(source_files.index(src) + 1).zfill(2)
    source_hash = sha256_file(src)
    for page_no, page_text in doc['pages']:
        if not page_text:
            continue
        for chunk_no, chunk in enumerate(paragraph_chunks(page_text, CHUNK_SIZE), 1):
            if chunk.strip():
                records.append({'record_id': f'{source_id}-P{page_no:04d}-C{chunk_no:03d}', 'source_id': source_id, 'source_filename': src.name, 'source_sha256': source_hash, 'page': page_no, 'chunk_id': chunk_no, 'text': chunk, 'processing': CHOSEN_METHOD})

if not records:
    raise ValueError('No prepared text chunks were produced.')
prepared_path = PROCESSED_DIR / 'prepared_text.jsonl'
with prepared_path.open('w', encoding='utf-8') as f:
    for row in records:
        f.write(json.dumps(row, ensure_ascii=False) + '\\n')
print('Prepared chunks:', len(records))
print('Prepared export:', prepared_path)
print('First prepared record:', json.dumps(records[0], ensure_ascii=False)[:800])


In [ ]:
# Cell 6 — Validate prepared export and source-to-output lineage
assert prepared_path.is_file() and prepared_path.stat().st_size > 0
with prepared_path.open('r', encoding='utf-8') as f:
    exported = [json.loads(line) for line in f if line.strip()]
assert len(exported) == len(records), 'Export count differs from in-memory record count.'
required = {'record_id','source_id','source_filename','source_sha256','page','chunk_id','text','processing'}
for i, row in enumerate(exported):
    assert required.issubset(row), f'Missing metadata in record {i}'
    assert row['text'].strip(), f'Empty text in record {i}'
    assert len(row['source_sha256']) == 64, f'Invalid source hash in record {i}'
    assert row['page'] >= 1
output_hash = sha256_file(prepared_path)
print('VALIDATION: PASS')
print('Records:', len(exported), '| output bytes:', prepared_path.stat().st_size)
print('Prepared output SHA-256:', output_hash)


In [ ]:
# Cell 7 — Write pipeline manifest with source and processing lineage
manifest = {
    'pipeline_version': 'utm-advisor-week3-v1',
    'generated_at_utc': datetime.now(timezone.utc).isoformat(),
    'project': 'UTM Academic Advisor',
    'selected_pathway': 'A_pdf_text',
    'chunking': {'method': CHOSEN_METHOD, 'chunk_size_characters': CHUNK_SIZE, 'page_boundaries_preserved': True},
    'sources': [{'source_id': 'S'+str(i+1).zfill(2), 'filename': d['path'].name, 'bytes': d['path'].stat().st_size, 'sha256': sha256_file(d['path']), 'pages': d['page_count'], 'extracted_characters': len(d['full_text']), 'empty_text_pages': d['empty_pages']} for i,d in enumerate(documents)],
    'outputs': [{'filename': prepared_path.name, 'records': len(records), 'bytes': prepared_path.stat().st_size, 'sha256': output_hash}],
    'limitations': ['PDF text extraction may not read scanned/image-only pages.', 'No OCR, embeddings, vector index or model inference is performed in this Topic 3 pipeline.']
}
manifest_path = PROCESSED_DIR / 'pipeline_manifest.json'
manifest_path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding='utf-8')
print('Manifest saved:', manifest_path)
print(json.dumps(manifest, indent=2, ensure_ascii=False)[:3000])


In [ ]:
# Cell 8 — Repeatability smoke test
def build_records_again():
    rerun = []
    for doc in documents:
        src = doc['path']; source_id = 'S' + str(source_files.index(src) + 1).zfill(2); source_hash = sha256_file(src)
        for page_no, page_text in doc['pages']:
            if not page_text: continue
            for chunk_no, chunk in enumerate(paragraph_chunks(page_text, CHUNK_SIZE), 1):
                if chunk.strip():
                    rerun.append({'record_id': f'{source_id}-P{page_no:04d}-C{chunk_no:03d}', 'source_id': source_id, 'source_filename': src.name, 'source_sha256': source_hash, 'page': page_no, 'chunk_id': chunk_no, 'text': chunk, 'processing': CHOSEN_METHOD})
    return rerun

rerun_records = build_records_again()
first_fingerprint = hashlib.sha256('\\n'.join(json.dumps(r, sort_keys=True, ensure_ascii=False) for r in records).encode('utf-8')).hexdigest()
second_fingerprint = hashlib.sha256('\\n'.join(json.dumps(r, sort_keys=True, ensure_ascii=False) for r in rerun_records).encode('utf-8')).hexdigest()
assert first_fingerprint == second_fingerprint, 'Repeatability check failed: prepared records differ.'
assert len(records) == len(rerun_records)
print('PASS: source hashes, record count, page/chunk mapping and prepared content are repeatable.')
print('Prepared-record fingerprint:', first_fingerprint)
print('NOTE: this is a same-runtime smoke test, not cross-environment reproducibility.')


## Output checklist
- `data/processed/prepared_text.jsonl` — prepared chunks with source/page metadata.
- `data/processed/pipeline_manifest.json` — source hashes, extraction summary and output fingerprint.

**Not performed in Topic 3:** embeddings, vector database construction, LLM generation or answer evaluation. These belong to later implementation stages.

After running, download both output files from the Colab Files panel or copy them to an authorised persistent location. Do not commit the original regulation PDFs unless redistribution is explicitly permitted.
